# Hiberno-English extraction pipeline

This notebook runs the extraction pipeline end to end, stage by stage.
Each stage names what it does and which library function does it.

Run top to bottom. Every stage that reads a file only reads from
`inputs/` (real inputs you supply locally -- see `inputs/README.md`) or
`data/<stage>/` (this notebook's own previous output).

**Status:** stages are added here one at a time as the repository is
built -- this is not yet the complete pipeline.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
INPUTS = REPO_ROOT / "inputs"
DATA = REPO_ROOT / "data"
sys.path.insert(0, str(SRC))

for d in ("01_ingest",):
    (DATA / d).mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("inputs/ present:", INPUTS.exists())

REPO_ROOT: d:\Experiments\Hiberno_english_Dictonary\Hiberno_english_Dictonary_Springer\public_release\hiberno-english-grounded-reader
inputs/ present: True


## Stage 0 -- Source document

The whole extraction pipeline operates on one witness document. It is
copyrighted and not shipped in this repository; place your own copy at
`inputs/source_witness.docx` (see `inputs/README.md`).

In [2]:
SOURCE_DOCX = INPUTS / "source_witness.docx"
if not SOURCE_DOCX.exists():
    raise FileNotFoundError(
        f"{SOURCE_DOCX} not found -- see inputs/README.md and place your "
        "own copy of the source document there before continuing.")

print("Found:", SOURCE_DOCX)

Found: d:\Experiments\Hiberno_english_Dictonary\Hiberno_english_Dictonary_Springer\public_release\hiberno-english-grounded-reader\inputs\source_witness.docx


## Stage 1 -- OOXML ingest

Reads the `.docx` as a raw zip/XML package and emits one row per
text-bearing run, in document order, with the *effective* character
formatting resolved through Word's real inheritance chain (document
defaults -> paragraph style -> character style -> direct formatting).
Several of those properties (italic, bold, caps, small caps, strike)
are toggles under the OOXML spec: where two levels of the hierarchy
each set one, the effective value is their XOR, not just the nearest
one -- this project's source relies on exactly that behaviour in
places, so resolving it correctly (not approximately) matters.

Before returning, it independently recounts every text node and
character in the raw XML and checks the emitted rows match exactly;
a mismatch raises `ConservationError` rather than returning a partial
result. This is the only stage that reads the raw document XML --
everything downstream works from this run table.

Function: `hiberno_english.ingest.ingest_docx`.
Output: `data/01_ingest/runs.parquet` (gitignored -- see `inputs/README.md`;
its `text` column reconstructs the document body verbatim).

In [3]:
from hiberno_english.ingest import ingest_docx, write_runs_parquet

ingest_result = ingest_docx(SOURCE_DOCX)
report = ingest_result.conservation

print("conservation_pass:", report.conservation_pass)
print("expected_text_nodes / emitted_rows:", report.expected_text_nodes, "/", report.emitted_rows)
print("expected_chars / emitted_chars:", report.expected_chars, "/", report.emitted_chars)
print("character_styles_defined:", report.character_styles_defined)
assert report.conservation_pass, "conservation gate failed -- see report above"

runs = ingest_result.runs
write_runs_parquet(runs, DATA / "01_ingest" / "runs.parquet")
print("runs table:", runs.num_rows, "rows ->", DATA / "01_ingest" / "runs.parquet")

conservation_pass: True
expected_text_nodes / emitted_rows: 24641 / 24641
expected_chars / emitted_chars: 917946 / 917946
character_styles_defined: 10
runs table: 24641 rows -> d:\Experiments\Hiberno_english_Dictonary\Hiberno_english_Dictonary_Springer\public_release\hiberno-english-grounded-reader\data\01_ingest\runs.parquet
